
<a href="https://colab.research.google.com/github/Jose-Bautista-gnss/Evaluacion_Grupal_1/blob/main/codigo/evaluacion_grupal_1_colab.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Google Colab"/>
</a>


# Homicidios por departamento en Colombia, 2025

Evaluación grupal: mapa de densidad de puntos (DDM), mapa de símbolos proporcionales (PSM) y mapa coroplético. En los tres mapas se utiliza la misma variable: `homicidios_2025`.

## 1. Librerías y datos desde GitHub

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = ["geopandas", "pyogrio", "mapclassify"]
missing_packages = [
    package for package in required_packages
    if importlib.util.find_spec(package) is None
]

if missing_packages:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing_packages
    ])

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve
import re
import unicodedata

import geopandas as gpd
import mapclassify
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.rcParams["figure.dpi"] = 120

In [ ]:
github_data = (
    "https://raw.githubusercontent.com/"
    "Jose-Bautista-gnss/Evaluacion_Grupal_1/main/data/"
)

gpkg_url = github_data + "departamentos.gpkg"
homicidios_url = github_data + "homicidios_2025.csv"

temporary_folder = TemporaryDirectory()
gpkg_path = Path(temporary_folder.name) / "departamentos.gpkg"
urlretrieve(gpkg_url, gpkg_path)

departamentos = gpd.read_file(gpkg_path, layer="departamento")
homicidios = pd.read_csv(homicidios_url, encoding="utf-8-sig")

print("Departamentos del mapa:", len(departamentos))
print("Filas de homicidios:", len(homicidios))

## 2. Unión del mapa con la variable

In [ ]:
def limpiar_nombre(valor):
    texto = str(valor).strip().upper()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caracter for caracter in texto
        if not unicodedata.combining(caracter)
    )
    texto = re.sub(r"[^A-Z0-9 ]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()

    equivalencias = {
        "MAGDANELA": "MAGDALENA",
        "SAN ANDRES": "SAN ANDRES Y PROVIDENCIA",
    }
    return equivalencias.get(texto, texto)


departamentos["departamento_clave"] = departamentos["NOM_DPTO"].map(limpiar_nombre)
homicidios["departamento_clave"] = homicidios["departamento"].map(limpiar_nombre)

colombia = departamentos.merge(
    homicidios[["departamento_clave", "homicidios_2025"]],
    on="departamento_clave",
    how="inner",
    validate="one_to_one",
)

if len(colombia) != 32:
    raise ValueError(f"La unión debe producir 32 departamentos, pero produjo {len(colombia)}.")

colombia = colombia.to_crs(epsg=9377)
colombia[["NOM_DPTO", "homicidios_2025"]].head()

## 3. Mapa de densidad de puntos (DDM)

Cada punto representa aproximadamente 10 homicidios.

In [ ]:
valor_punto = 10
colombia["numero_puntos"] = np.maximum(
    1,
    np.rint(colombia["homicidios_2025"] / valor_punto).astype(int),
)

puntos_por_departamento = colombia.geometry.sample_points(
    size=colombia["numero_puntos"],
    rng=2025,
)

puntos = gpd.GeoDataFrame(
    geometry=puntos_por_departamento.explode(index_parts=False),
    crs=colombia.crs,
)

fig, ax = plt.subplots(figsize=(8, 9))

colombia.plot(
    ax=ax,
    facecolor="white",
    edgecolor="#555555",
    linewidth=0.5,
)
puntos.plot(
    ax=ax,
    color="#d7301f",
    markersize=1.6,
)

leyenda = Line2D(
    [0], [0],
    marker="o",
    linestyle="none",
    markerfacecolor="#d7301f",
    markeredgecolor="none",
    markersize=5,
    label="1 punto ≈ 10 homicidios",
)

ax.legend(handles=[leyenda], loc="lower left", frameon=False)
ax.set_title("Homicidios registrados en Colombia durante 2025 - DDM")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4. Diagrama de cajas

El diagrama permite identificar los departamentos con valores atípicos antes de elaborar el PSM.

In [ ]:
q1 = colombia["homicidios_2025"].quantile(0.25)
q3 = colombia["homicidios_2025"].quantile(0.75)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

colombia["atipico"] = colombia["homicidios_2025"] > limite_superior

fig, ax = plt.subplots(figsize=(8, 3))
ax.boxplot(
    colombia["homicidios_2025"],
    vert=False,
    patch_artist=True,
    boxprops={"facecolor": "#9ecae1"},
    medianprops={"color": "#08519c", "linewidth": 2},
    flierprops={
        "marker": "o",
        "markerfacecolor": "#f16913",
        "markeredgecolor": "#d94801",
    },
)
ax.set_title("Distribución de homicidios por departamento")
ax.set_xlabel("Homicidios registrados en 2025")
ax.set_yticks([])
plt.tight_layout()
plt.show()

colombia.loc[
    colombia["atipico"],
    ["NOM_DPTO", "homicidios_2025"],
].sort_values("homicidios_2025", ascending=False)

## 5. Mapa de símbolos proporcionales (PSM)

In [ ]:
simbolos = colombia.copy()
simbolos["geometry"] = simbolos.representative_point()

escala_simbolo = 0.35
simbolos["tamano"] = simbolos["homicidios_2025"] * escala_simbolo

fig, ax = plt.subplots(figsize=(8, 9))

colombia.plot(
    ax=ax,
    facecolor="white",
    edgecolor="#555555",
    linewidth=0.5,
)

simbolos_regulares = simbolos[~simbolos["atipico"]]
simbolos_atipicos = simbolos[simbolos["atipico"]]

simbolos_regulares.plot(
    ax=ax,
    markersize=simbolos_regulares["tamano"],
    color="#3182bd",
    edgecolor="#08519c",
    linewidth=0.5,
    alpha=0.70,
)
simbolos_atipicos.plot(
    ax=ax,
    markersize=simbolos_atipicos["tamano"],
    color="#fdae6b",
    edgecolor="#e6550d",
    linewidth=0.6,
    alpha=0.80,
)

valores_leyenda = [100, 1000, 3000]
simbolos_leyenda = [
    ax.scatter(
        [], [],
        s=valor * escala_simbolo,
        color="#3182bd",
        edgecolor="#08519c",
        alpha=0.70,
        label=f"{valor:,}".replace(",", " "),
    )
    for valor in valores_leyenda
]

leyenda_tamano = ax.legend(
    handles=simbolos_leyenda,
    title="Homicidios",
    loc="lower left",
    frameon=False,
)
ax.add_artist(leyenda_tamano)

leyenda_color = [
    Patch(facecolor="#3182bd", edgecolor="#08519c", label="Valor regular"),
    Patch(facecolor="#fdae6b", edgecolor="#e6550d", label="Valor atípico"),
]
ax.legend(
    handles=leyenda_color,
    title="Clasificación",
    loc="upper right",
    frameon=False,
)
ax.set_title("Homicidios registrados en Colombia durante 2025 - PSM")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 6. Comparación de métodos de clasificación

Se comparan seis métodos con cinco clases. Se selecciona el método con menor ADCM, porque reúne valores más semejantes dentro de cada clase.

In [ ]:
np.random.seed(2025)

k = 5
variable = colombia["homicidios_2025"]

clasificadores = [
    mapclassify.EqualInterval(variable, k=k),
    mapclassify.Quantiles(variable, k=k),
    mapclassify.MaximumBreaks(variable, k=k),
    mapclassify.FisherJenks(variable, k=k),
    mapclassify.JenksCaspall(variable, k=k),
    mapclassify.NaturalBreaks(variable, k=k),
]

comparacion_adcm = pd.DataFrame({
    "Clasificador": [clasificador.name for clasificador in clasificadores],
    "ADCM": [clasificador.adcm for clasificador in clasificadores],
}).sort_values("ADCM").reset_index(drop=True)

comparacion_adcm

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

comparacion_adcm.sort_values("ADCM", ascending=False).plot.barh(
    x="Clasificador",
    y="ADCM",
    legend=False,
    color="#756bb1",
    ax=ax,
)

ax.set_title("Comparación de métodos de clasificación")
ax.set_xlabel("ADCM (menor es mejor)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
clasificador_elegido = min(
    clasificadores,
    key=lambda clasificador: clasificador.adcm,
)

print("Método elegido:", clasificador_elegido.name)
print("ADCM:", round(clasificador_elegido.adcm, 2))

colombia["codigo_clase"] = clasificador_elegido.yb

etiquetas = {
    0: "Muy bajo",
    1: "Bajo",
    2: "Medio",
    3: "Alto",
    4: "Muy alto",
}

colombia["clase_homicidios"] = pd.Categorical(
    colombia["codigo_clase"].map(etiquetas),
    categories=list(etiquetas.values()),
    ordered=True,
)

pd.DataFrame({
    "Clase": list(etiquetas.values()),
    "Límite superior": np.round(clasificador_elegido.bins, 2),
})

## 7. Mapa coroplético

La coropleta utiliza las cinco clases del método que obtuvo el menor ADCM.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))

colombia.plot(
    column="clase_homicidios",
    categorical=True,
    cmap="YlOrRd",
    edgecolor="#555555",
    linewidth=0.5,
    legend=True,
    legend_kwds={
        "title": "Homicidios",
        "loc": "lower left",
        "frameon": False,
    },
    ax=ax,
)

ax.set_title(
    "Homicidios registrados en Colombia durante 2025 - Coropleta\n"
    f"Método: {clasificador_elegido.name}"
)
ax.set_axis_off()
plt.tight_layout()
plt.show()